In [3]:
import pandas as pd
import pulp
import numpy as np

print("--- Initializing Single-Objective Deterministic MILP ---")

# data preparation and variable initialization
print("Loading data and calculating absolute block values...")
df = pd.read_csv('block_model.csv')

# Convert rates/percentages into absolute totals for the MILP solver
df['total_npv'] = df['npv_per_t'] * (df['tonnage_kt'] * 1000)
df['total_ghg'] = df['ghg_kg_co2_per_t'] * df['tonnage_kt'] # Metric tonnes CO2
df['total_waste_kt'] = df['tonnage_kt'] * df['waste_ratio']

blocks = df['block_id'].tolist()
periods = [1, 2, 3, 4, 5] # 5-period life-of-mine schedule

# calculate 3D precedence cones for each block (i.e., which blocks must be mined before others)
print("Calculating 3D spatial precedence cones (Laws of Physics)...")
predecessors = {b: [] for b in blocks}

x_step = 50 # Example block width in X axis
y_step = 50 # Example block width in Y axis
z_step = 10 # Example bench height in Z axis

for idx, row in df.iterrows():
    b_id = row['block_id']
    bx, by, bz = row['x'], row['y'], row['z']
    
    # Find blocks sitting directly on top and adjacent in the bench above (z + z_step)
    above_blocks = df[
        (df['z'] == bz + z_step) & 
        (df['x'] >= bx - x_step) & (df['x'] <= bx + x_step) &
        (df['y'] >= by - y_step) & (df['y'] <= by + y_step)
    ]['block_id'].tolist()
    
    predecessors[b_id] = above_blocks

# model initialization
print("Building MILP variables and equations...")
model = pulp.LpProblem("Mine_Scheduling_Max_NPV", pulp.LpMaximize)

# Decision Variable: x[b, t] = 1 if block 'b' is mined in period 't', 0 otherwise
x = pulp.LpVariable.dicts("mine_block",
                          [(b, t) for b in blocks for t in periods],
                          cat='Binary')

# objective function (equation 1)
# Maximize Total Mine NPV
model += pulp.lpSum([df.loc[df['block_id'] == b, 'total_npv'].values[0] * x[(b, t)] 
                     for b in blocks for t in periods]), "Total_NPV"


# CONSTRAINTS (Equations 4, 5, 6, 7 & Carbon)


# Eq 4: Reserve Constraint (A block can only be mined once, or left behind)
for b in blocks:
    model += pulp.lpSum([x[(b, t)] for t in periods]) <= 1, f"Reserve_{b}"

# Eq 5: Precedence Constraint (Cannot mine a block until dirt above it is removed)
for b in blocks:
    for t in periods:
        for p in predecessors[b]:
            model += x[(b, t)] - pulp.lpSum([x[(p, tau)] for tau in range(1, t + 1)]) <= 0, f"Prec_{b}_{p}_t{t}"

# Week 4 Custom: Carbon Emissions Budget per period
carbon_budget_per_period = 50_000_000 # Max 1.5 Million tonnes CO2 per period
for t in periods:
    model += pulp.lpSum([df.loc[df['block_id'] == b, 'total_ghg'].values[0] * x[(b, t)] 
                         for b in blocks]) <= carbon_budget_per_period, f"Carbon_Budget_{t}"

# Eq 6 & 7: Operational Capacities
M_min, M_max = 0, 100_000 # Total mining limits (ore + waste) in kt
P_min, P_max = 0, 50_000  # Processing limits (ore only) in kt

cutoff_grade = 0.5 # Define what constitutes "ore"
ore_blocks = df[df['ore_grade_pct'] >= cutoff_grade]['block_id'].tolist()

for t in periods:
    # Mining Limits (All material)
    model += pulp.lpSum([df.loc[df['block_id'] == b, 'tonnage_kt'].values[0] * x[(b, t)] 
                         for b in blocks]) >= M_min, f"Min_Mine_{t}"
    model += pulp.lpSum([df.loc[df['block_id'] == b, 'tonnage_kt'].values[0] * x[(b, t)] 
                         for b in blocks]) <= M_max, f"Max_Mine_{t}"
    
    # Processing Limits (Ore only)
    model += pulp.lpSum([df.loc[df['block_id'] == b, 'tonnage_kt'].values[0] * x[(b, t)] 
                         for b in ore_blocks]) >= P_min, f"Min_Proc_{t}"
    model += pulp.lpSum([df.loc[df['block_id'] == b, 'tonnage_kt'].values[0] * x[(b, t)] 
                         for b in ore_blocks]) <= P_max, f"Max_Proc_{t}"

# execute the solver
print("Handing model over to PuLP Solver... (This may take a moment)")
model.solve()

# results output
print(f"\n--- SOLVER STATUS: {pulp.LpStatus[model.status]} ---")

if model.status == pulp.LpStatusOptimal:
    optimal_npv = pulp.value(model.objective)
    print(f"Optimal Mine NPV (Subject to strict carbon laws): ${optimal_npv / 1e9:.2f} Billion")
    
    # Calculate exactly how much carbon was emitted to achieve this NPV
    total_carbon_emitted = sum(
        df.loc[df['block_id'] == b, 'total_ghg'].values[0] * x[(b, t)].varValue
        for b in blocks for t in periods if x[(b, t)].varValue == 1.0
    )
    print(f"Total Life-of-Mine Carbon Footprint: {total_carbon_emitted / 1e6:.2f} Million Tonnes")
else:
    print("Solver could not find an optimal solution. Try relaxing the Carbon Budget or Capacity Constraints.")

--- Initializing Single-Objective Deterministic MILP ---
Loading data and calculating absolute block values...
Calculating 3D spatial precedence cones (Laws of Physics)...
Building MILP variables and equations...
Handing model over to PuLP Solver... (This may take a moment)

--- SOLVER STATUS: Optimal ---
Optimal Mine NPV (Subject to strict carbon laws): $17.03 Billion
Total Life-of-Mine Carbon Footprint: 6.89 Million Tonnes
